In [0]:
spark.conf.set(
    "fs.azure.account.key.stdeltalakedemo.dfs.core.windows.net",
    "00000000000000000000000000000000000000000000000000000000000000000000000000000000000000==")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4949399039578160>, line 1
----> 1 spark.conf.set(
      2     "fs.azure.account.key.stdeltalakedemo.dfs.core.windows.net",
      3     "v4ZQzonO9lggJo58S6JOeYAbtr3e+ZiK0jPApirUwzOoySbME54WFOcsKxjjupxzjZ0cuUESGSyI+AStJUKNqg==")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:46, in RuntimeConf.set(self, key, value)
     44 op_set = proto.ConfigRequest.Set(pairs=[proto.KeyValue(key=key, value=value)])
     45 operation = proto.ConfigRequest.Operation(set=op_set)
---> 46 result = self._client.config(operation)
     47 for warn in result.warnings:
     48     warnings.warn(warn)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2193, in SparkConnectClient.config(self, operation)
   2191     raise SparkConnectException("Invalid state during retry exception

In [0]:
# Define widgets (ADF will inject values here)
dbutils.widgets.text("p_trainee_id","temp")
dbutils.widgets.text("p_input_path","temp")
dbutils.widgets.text("p_bronze_path","temp")

# Get parameter values
p_trainee_id = dbutils.widgets.get("p_trainee_id")
input_path   = dbutils.widgets.get("p_input_path")
bronze_path  = dbutils.widgets.get("p_bronze_path")

 
# Read stream
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", 
                f"abfss://checkpoint@stdeltalakedemo.dfs.core.windows.net/{p_trainee_id}/schema")
        .load(input_path)
)

# Write stream
(
    df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            f"abfss://checkpoint@stdeltalakedemo.dfs.core.windows.net/{p_trainee_id}/"
        )
        .trigger(availableNow=True)
        .start(bronze_path)
        .awaitTermination()
)


---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-4949399039577624>, line 32
     15 df = (
     16     spark.readStream
     17         .format("cloudFiles")
   (...)
     20         .load(input_path)
     21 )
     23 # Write stream
     24 (
     25     df.writeStream
     26         .format("delta")
     27         .option(
     28             "checkpointLocation",
     29             f"abfss://checkpoint@stdeltalakedemo.dfs.core.windows.net/{p_trainee_id}/"
     30         )
     31         .trigger(availableNow=True)
---> 32         .start(bronze_path)
     33         .awaitTermination()
     34 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:723, in DataStreamWriter.start(self, path, format, outputMode, partitionBy, queryName, **options)
    714 def start(
    715     self,
    716     path: Optional[str] = None,


In [0]:
# print(p_trainee_id, input_path)

rookie04 abfss://landing@stdeltalakedemo.dfs.core.windows.net/rookie04


In [0]:
# display(df)